# درس ۱۳ - حافظه عامل


## راه‌اندازی

این دفترچه یادداشت نشان می‌دهد چگونه یک عامل رزرو سفر با **حافظه پایدار** با استفاده از **چارچوب عامل مایکروسافت** (MAF) بسازیم.

شما خواهید آموخت که چگونه انواع مختلف حافظه عامل — کاری، کوتاه‌مدت و بلندمدت — بر نحوه حفظ و استفاده عامل از اطلاعات در طول گفتگوها تأثیر می‌گذارند.

**پیش‌نیازها:**
- یک پروژه Microsoft Foundry با مدل چت مستقر شده (مثلاً `gpt-5-mini`).
- وارد شده با Azure CLI — اجرای `az login` در ترمینال شما.
- `AZURE_AI_PROJECT_ENDPOINT` — نقطه پایانی پروژه Microsoft Foundry شما.
- `AZURE_AI_MODEL_DEPLOYMENT_NAME` — نام مدل مستقر شده شما.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import json
import dotenv
from typing import Annotated
from datetime import datetime

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

print("Microsoft Foundry client configured")

## انواع حافظه عامل

عوامل هوش مصنوعی می‌توانند از انواع مختلف حافظه استفاده کنند که هرکدام هدفی متفاوت را دنبال می‌کنند:

### حافظه کاری
خود رشته گفتگو — پیام‌های رد و بدل شده در یک جلسه واحد. عامل می‌تواند برای حفظ انسجام به پیام‌های قبلی در همان رشته مراجعه کند. در MAF این حافظه از طریق **`agent.create_session()`** ایجاد می‌شود که یک `AgentSession` برمی‌گرداند.

### حافظه کوتاه‌مدت
اطلاعاتی که برای مدت زمان یک کار یا جلسه باقی می‌ماند اما به صورت دائمی ذخیره نمی‌شود. به عنوان مثال، عامل ممکن است در طول یک گفتگوی برنامه‌ریزی چند مرحله‌ای حقایق را جمع‌آوری کند و از آن‌ها برای تولید برنامه نهایی استفاده کند.

### حافظه بلندمدت
ترجیحات و حقایقی که **در طول جلسات مختلف** باقی می‌مانند. کاربران بازگشتی نباید مجبور باشند محدودیت‌های غذایی یا سبک سفر خود را دوباره تکرار کنند. حافظه بلندمدت معمولاً توسط یک ذخیره‌ساز خارجی پشتیبانی می‌شود — یک پایگاه داده، فایل، یا شاخص برداری — و از طریق ابزارها به عامل ارائه می‌شود.


## حافظه کاری با جلسات

ساده‌ترین شکل حافظه، جلسه مکالمه است. وقتی همان جلسه (ایجادشده از طریق `agent.create_session()`) را به فراخوانی‌های متوالی `agent.run()` منتقل می‌کنید، عامل کل تاریخچه آن مکالمه را می‌بیند و می‌تواند جزئیات قبلی را به خاطر بسپارد.

بیایید یک عامل مسافرتی ایجاد کنیم و حافظه کاری را نمایش دهیم.


In [ ]:
agent = client.as_agent(
    name="TravelMemoryAgent",
    instructions=(
        "You are a travel agent who remembers user preferences across conversations. "
        "Track destinations mentioned, budget constraints, and travel dates."
    ),
)

session = agent.create_session()

# First message — the user shares preferences
response = await agent.run(
    "I love beach destinations and my budget is $3000",
    session=session,
)
print("Agent:", response)

# Second message — the agent should recall the budget from the thread
response = await agent.run(
    "What did I say my budget was?",
    session=session,
)
print("Agent:", response)

عامل به‌درستی بودجه را به خاطر سپرد زیرا هر دو پیام از همان جلسه مشترک هستند. این همان **حافظه کاری** است — که فقط در طول عمر جلسه وجود دارد.

### با یک رشته جدید چه اتفاقی می‌افتد؟

اگر یک جلسه **جدید** ایجاد کنیم، عامل هیچ حافظه‌ای از گفتگوی قبلی ندارد:


In [ ]:
new_session = agent.create_session()

response = await agent.run(
    "What is my budget?",
    session=new_session,
)
print("Agent:", response)
print("\n💡 The agent has no memory of the previous conversation — it's a fresh session.")

## الگوی حافظه بلندمدت

برای به یاد داشتن ترجیحات کاربر **در جلسات مختلف**، ما به یک حافظه پایدار نیاز داریم که بیرون از رشته گفتگو زنده بماند. عامل این حافظه را از طریق **ابزارها** — توابعی که می‌تواند برای ذخیره و بازیابی اطلاعات فراخوانی کند — دسترسی دارد.

در ادامه یک مخزن ترجیحات ساده در حافظه پیاده‌سازی می‌کنیم (در تولید شما این را با یک پایگاه داده یا شاخص برداری پشتیبانی می‌کنید) و آن را به صورت ابزارهایی که عامل می‌تواند استفاده کند، عرضه می‌کنیم.

### معماری
```
┌─────────────────┐     ┌──────────────────┐     ┌─────────────────┐
│  MAF Agent      │────▶│  @tool functions  │────▶│  Preference     │
│  (LLM)          │     │  save / retrieve  │     │  Store (dict)   │
└─────────────────┘     └──────────────────┘     └─────────────────┘
         │                                                 │
    AgentSession                                   Persists across
    (working memory)                               sessions
```


In [ ]:
# --- Persistent preference store (simulated) ---
preference_store: dict[str, list[str]] = {}


@tool(approval_mode="never_require")
def save_preference(
    user_id: Annotated[str, "User identifier"],
    preference: Annotated[str, "A travel preference to remember"],
) -> str:
    """Save a user travel preference to long-term memory."""
    preference_store.setdefault(user_id, []).append(preference)
    return f"✅ Stored: {preference}"


@tool(approval_mode="never_require")
def get_preferences(
    user_id: Annotated[str, "User identifier"],
) -> str:
    """Retrieve all saved travel preferences for a user."""
    prefs = preference_store.get(user_id, [])
    if not prefs:
        return f"No saved preferences for {user_id}."
    return "Saved preferences:\n- " + "\n- ".join(prefs)


@tool(approval_mode="never_require")
def search_hotels(
    query: Annotated[str, "Search query — location, amenities, or tags"],
) -> str:
    """Search the hotel database for matching properties."""
    hotels = [
        {"name": "Le Meurice Paris", "location": "Paris, France", "price": 850, "tags": ["luxury", "romantic", "spa"]},
        {"name": "Four Seasons Maui", "location": "Maui, Hawaii", "price": 695, "tags": ["beach", "family", "resort"]},
        {"name": "Aman Tokyo", "location": "Tokyo, Japan", "price": 780, "tags": ["luxury", "city", "spa"]},
        {"name": "Hotel Sacher Vienna", "location": "Vienna, Austria", "price": 420, "tags": ["historic", "accessible", "cultural"]},
        {"name": "Fairmont Whistler", "location": "Whistler, Canada", "price": 380, "tags": ["ski", "family", "mountain"]},
    ]
    q = query.lower()
    matches = [
        h for h in hotels
        if q in h["name"].lower()
        or q in h["location"].lower()
        or any(q in t for t in h["tags"])
    ]
    if not matches:
        matches = hotels[:3]
    return json.dumps(matches, indent=2)


print("✅ Tools defined: save_preference, get_preferences, search_hotels")

### سناریو 1 — کاربر برای اولین بار سفر سالگرد را رزرو می‌کند

سارا برای اولین بار بازدید می‌کند. نماینده باید ترجیحات او را از طریق ابزارها ذخیره کرده و از آن‌ها برای پیشنهاد هتل‌ها استفاده کند.


In [ ]:
travel_agent = client.as_agent(
    tools=[save_preference, get_preferences],
    name="TravelBookingAssistant",
    instructions=(
        "You are a personalized travel booking assistant with long-term memory.\n"
        "WORKFLOW:\n"
        "1. When a user starts a conversation, call get_preferences() to check for saved information.\n"
        "2. Store any new preferences the user mentions using save_preference().\n"
        "3. Use search_hotels() to find suitable options that match their preferences and budget.\n"
        "4. Do NOT recommend hotels that exceed the user's budget.\n\n"
        "IMPORTANT: Always use user_id='sarah_johnson_123' for all memory operations."
    ),
)

session_1 = travel_agent.create_session()

response = await travel_agent.run(
    "Hi! I'm Sarah and I'm planning a trip for my 10th wedding anniversary. "
    "We love romantic destinations, fine dining, and spa experiences. "
    "My husband has mobility issues, so we need accessible accommodations. "
    "Our budget is around $700-800 per night.",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
response = await travel_agent.run(
    "The Hotel Sacher sounds perfect! We're both vegetarian and I have a "
    "severe nut allergy. Can you note that for future trips?",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
# Verify what was stored
print("📋 Preference store contents:")
for uid, prefs in preference_store.items():
    print(f"\n  User: {uid}")
    for p in prefs:
        print(f"    - {p}")

### سناریو ۲ — سارا چند هفته بعد بازمی‌گردد

سارا یک **رشته جدید** شروع می‌کند (شبیه‌سازی یک نشست جدید). حافظه کاری خالی است، اما فروشگاه اولویت‌های بلندمدت هنوز اطلاعات او را دارد. عامل باید آن را بازیابی کند و از آن برای شخصی‌سازی پیشنهادها استفاده کند.


In [ ]:
session_2 = travel_agent.create_session()  # New session — no working memory

response = await travel_agent.run(
    "Hi, my husband and I are planning another trip. Can you recommend a good hotel?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 The agent retrieved Sarah's saved preferences from long-term memory "
      "even though this is a completely new conversation thread.")

In [ ]:
response = await travel_agent.run(
    "Great suggestions! For the Maui option, what activities would you recommend for the kids?",
    session=session_2,
)
print("🤖 Agent:", response)

## خلاصه

در این درس با سه نوع حافظه عامل و نحوه پیاده‌سازی آن‌ها با استفاده از چارچوب عامل مایکروسافت آشنا شدید:

| نوع حافظه | مکانیزم MAF | مدت زمان |
|---|---|---|
| **کاری** | `agent.create_session()` | یک مکالمه واحد |
| **کوتاه‌مدت** | زمینه انباشته شده در یک رشته | یک وظیفه / جلسه واحد |
| **بلندمدت** | ذخیره خارجی که از طریق توابع `@tool` دسترسی دارد | میان جلسات |

### کلید‌های مهم
1. **`agent.create_session()`** حافظه کاری را فراهم می‌کند — عامل کل تاریخچه مکالمه را در یک جلسه می‌بیند.
2. **جلسات جدید زمینه را از دست می‌دهند** — بدون حافظه بلندمدت عامل نمی‌تواند مکالمات گذشته را به خاطر بیاورد.
3. **توابع `@tool`** پل بین این‌ها هستند — به عامل اجازه می‌دهند اطلاعات را در یک ذخیره پایدار ذخیره و بازیابی کند.
4. **شخصی‌سازی با گذر زمان بهبود می‌یابد** — هر چه بیشتر ترجیحات ذخیره شود، پیشنهادهای عامل بهتر خواهد بود.

### کاربردهای دنیای واقعی
- **خدمات مشتری**: به خاطر سپردن تاریخچه و ترجیحات مشتری
- **دستیارهای شخصی**: حفظ زمینه در طول روزها یا هفته‌ها
- **بهداشت و درمان**: پیگیری اطلاعات و ترجیحات بیمار
- **تجارت الکترونیک**: خرید شخصی‌سازی شده بر اساس تاریخچه

### گام‌های بعدی
- جایگزینی دیکشنری در حافظه با یک پایگاه داده یا ذخیره برداری (مثلاً Azure AI Search)
- افزودن انقضای حافظه برای اطلاعات حساس به زمان
- ساخت سیستم‌های چندعاملی با حافظه مشترک
- بررسی یادداشت‌بردار Cognee برای حافظه پشتیبانی‌شده از نمودار دانش


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**سلب مسئولیت**:
این سند با استفاده از سرویس ترجمه هوش مصنوعی [Co-op Translator](https://github.com/Azure/co-op-translator) ترجمه شده است. در حالی که ما در تلاش برای دقت هستیم، لطفاً توجه داشته باشید که ترجمه‌های خودکار ممکن است شامل خطاها یا نادرستی‌هایی باشند. سند اصلی به زبان مادری خود باید به عنوان منبع معتبر در نظر گرفته شود. برای اطلاعات حیاتی، ترجمه حرفه‌ای انسانی توصیه می‌شود. ما در قبال هرگونه سوء تفاهم یا برداشت نادرست ناشی از استفاده از این ترجمه مسئولیتی نداریم.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
